In [6]:
import pandas as pd
import numpy as np
import importlib
import sys

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix

from src.preprocessing import load_and_preprocess_data
from src.models import (
    train_logistic_regression,
    train_decision_tree,
    train_knn,
    train_neural_network
)
from src.evaluation import evaluate_model
from sklearn.model_selection import GridSearchCV
import pandas as pd

In [7]:
print("Starting Data Preprocessing Pipeline")

dataset_path = "archive/KDDTrain+.txt"

X_train, X_test, y_train, y_test = load_and_preprocess_data(dataset_path)

print("\nPreprocessing Completed Successfully!")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print("\nTarget label distribution (Training Set):")
print(y_train.value_counts())

Starting Data Preprocessing Pipeline

Preprocessing Completed Successfully!
X_train shape: (100778, 119)
X_test shape: (25195, 119)

Target label distribution (Training Set):
label
0    53874
1    46904
Name: count, dtype: int64


In [9]:
from sklearn.linear_model import LogisticRegression

lr_param_grid = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l2"],
    "solver": ["lbfgs"]
}

lr_grid = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    param_grid=lr_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

lr_grid.fit(X_train, y_train)

print("Best LR params:", lr_grid.best_params_)
print("Best LR CV score:", lr_grid.best_score_)

best_lr = lr_grid.best_estimator_

Best LR params: {'C': 10, 'penalty': 'l2', 'solver': 'lbfgs'}
Best LR CV score: 0.9721921232058195


In [10]:
from sklearn.tree import DecisionTreeClassifier

dt_param_grid = {
    "max_depth": [5, 10, 15, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

dt_grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=dt_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

dt_grid.fit(X_train, y_train)

print("Best DT params:", dt_grid.best_params_)
print("Best DT CV score:", dt_grid.best_score_)

best_dt = dt_grid.best_estimator_

Best DT params: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best DT CV score: 0.9980169732230365


In [18]:
from sklearn.neighbors import KNeighborsClassifier

knn_param_grid = {
    "n_neighbors": [3, 5, 7, 9],
    "weights": ["uniform", "distance"],
    "p": [1, 2]
}

knn_grid = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=knn_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

knn_grid.fit(X_train, y_train)

print("Best KNN params:", knn_grid.best_params_)
print("Best KNN CV score:", knn_grid.best_score_)

best_knn = knn_grid.best_estimator_

Best KNN params: {'n_neighbors': 3, 'p': 1, 'weights': 'distance'}
Best KNN CV score: 0.9971734303024118


In [19]:
results = []

res_lr = evaluate_model(best_lr, X_test, y_test, "Logistic Regression (Tuned)")
results.append(res_lr)

res_dt = evaluate_model(best_dt, X_test, y_test, "Decision Tree (Tuned)")
results.append(res_dt)

res_knn = evaluate_model(best_knn, X_test, y_test, "KNN (Tuned)")
results.append(res_knn)


=== Logistic Regression (Tuned) ===
Accuracy:  97.23%
Precision: 97.63%
Recall:    96.39%
F1-Score:  97.01%
Confusion Matrix:
[[13195   274]
 [  423 11303]]
------------------------------

=== Decision Tree (Tuned) ===
Accuracy:  99.86%
Precision: 99.81%
Recall:    99.89%
F1-Score:  99.85%
Confusion Matrix:
[[13447    22]
 [   13 11713]]
------------------------------

=== KNN (Tuned) ===
Accuracy:  99.78%
Precision: 99.73%
Recall:    99.80%
F1-Score:  99.77%
Confusion Matrix:
[[13437    32]
 [   23 11703]]
------------------------------


In [20]:
nn1_model, nn1_history = train_neural_network(
    X_train,
    y_train,
    layers_config=[64, 32],
    learning_rate=0.001,
    dropout_rate=0.3,
    epochs=50,
    batch_size=64
)

res_nn1 = evaluate_model(
    nn1_model,
    X_test,
    y_test,
    "Neural Network A [64, 32]",
    threshold=0.5
)
results.append(res_nn1)

print("NN A epochs trained:", len(nn1_history.history["loss"]))
print("NN A final train acc:", nn1_history.history["accuracy"][-1])
print("NN A final val acc:", nn1_history.history["val_accuracy"][-1])

c:\Users\fland\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


788/788 ━━━━━━━━━━━━━━━━━━━━ 0s 499us/step

=== Neural Network A [64, 32] ===
Accuracy:  99.50%
Precision: 99.54%
Recall:    99.39%
F1-Score:  99.46%
Confusion Matrix:
[[13415    54]
 [   72 11654]]
------------------------------
NN A epochs trained: 22
NN A final train acc: 0.9950757622718811
NN A final val acc: 0.9944929480552673


In [21]:
nn2_model, nn2_history = train_neural_network(
    X_train,
    y_train,
    layers_config=[128, 64, 32],
    learning_rate=0.001,
    dropout_rate=0.3,
    epochs=50,
    batch_size=64
)

res_nn2 = evaluate_model(
    nn2_model,
    X_test,
    y_test,
    "Neural Network B [128, 64, 32]",
    threshold=0.5
)
results.append(res_nn2)

print("NN B epochs trained:", len(nn2_history.history["loss"]))
print("NN B final train acc:", nn2_history.history["accuracy"][-1])
print("NN B final val acc:", nn2_history.history["val_accuracy"][-1])

c:\Users\fland\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


788/788 ━━━━━━━━━━━━━━━━━━━━ 1s 624us/step

=== Neural Network B [128, 64, 32] ===
Accuracy:  99.48%
Precision: 99.54%
Recall:    99.33%
F1-Score:  99.44%
Confusion Matrix:
[[13415    54]
 [   78 11648]]
------------------------------
NN B epochs trained: 23
NN B final train acc: 0.9951130151748657
NN B final val acc: 0.9950386881828308


In [22]:
summary_rows = []

for r in results:
    summary_rows.append({
        "Model": r["Model"],
        "Accuracy": round(r["Accuracy"], 4),
        "Precision": round(r["Precision"], 4),
        "Recall": round(r["Recall"], 4),
        "F1": round(r["F1"], 4)
    })

results_df = pd.DataFrame(summary_rows)
results_df = results_df.sort_values(by="F1", ascending=False).reset_index(drop=True)

print("\nFinal Comparison Table:")
print(results_df.to_string(index=False))


Final Comparison Table:
                         Model  Accuracy  Precision  Recall     F1
         Decision Tree (Tuned)    0.9986     0.9981  0.9989 0.9985
                   KNN (Tuned)    0.9978     0.9973  0.9980 0.9977
     Neural Network A [64, 32]    0.9950     0.9954  0.9939 0.9946
Neural Network B [128, 64, 32]    0.9948     0.9954  0.9933 0.9944
   Logistic Regression (Tuned)    0.9723     0.9763  0.9639 0.9701


In [23]:
best_params_df = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Tested Values": "C=[0.01, 0.1, 1, 10], penalty=['l2'], solver=['lbfgs']",
        "Best Params": str(lr_grid.best_params_)
    },
    {
        "Model": "Decision Tree",
        "Tested Values": "max_depth=[5,10,15,None], min_samples_split=[2,5,10], min_samples_leaf=[1,2,4]",
        "Best Params": str(dt_grid.best_params_)
    },
    {
        "Model": "KNN",
        "Tested Values": "n_neighbors=[3,5,7,9], weights=['uniform','distance'], p=[1,2]",
        "Best Params": str(knn_grid.best_params_)
    },
    {
        "Model": "Neural Network A",
        "Tested Values": "layers=[64,32], lr=0.001, dropout=0.3, batch_size=64",
        "Best Params": "Manual architecture experiment"
    },
    {
        "Model": "Neural Network B",
        "Tested Values": "layers=[128,64,32], lr=0.001, dropout=0.3, batch_size=64",
        "Best Params": "Manual architecture experiment"
    }
])

print("\nBest Parameters Table:")
print(best_params_df.to_string(index=False))


Best Parameters Table:
              Model                                                                  Tested Values                                                        Best Params
Logistic Regression                         C=[0.01, 0.1, 1, 10], penalty=['l2'], solver=['lbfgs']                      {'C': 10, 'penalty': 'l2', 'solver': 'lbfgs'}
      Decision Tree max_depth=[5,10,15,None], min_samples_split=[2,5,10], min_samples_leaf=[1,2,4] {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
                KNN                 n_neighbors=[3,5,7,9], weights=['uniform','distance'], p=[1,2]                  {'n_neighbors': 3, 'p': 1, 'weights': 'distance'}
   Neural Network A                           layers=[64,32], lr=0.001, dropout=0.3, batch_size=64                                     Manual architecture experiment
   Neural Network B                       layers=[128,64,32], lr=0.001, dropout=0.3, batch_size=64                                     Manual arch

## Clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
X_all = np.vstack([X_train, X_test])
y_all = np.hstack([y_train.values, y_test.values])

print("Total data for clustering:", X_all.shape)

In [ ]:
k = 2

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X_all)

ari = adjusted_rand_score(y_all, clusters)
nmi = normalized_mutual_info_score(y_all, clusters)

print(f"\nKMeans Clustering (k={k})")
print(f"Adjusted Rand Index (ARI): {ari:.4f}")
print(f"Normalized Mutual Information (NMI): {nmi:.4f}")

In [ ]:
cluster_results = []

for k in [2, 3, 5]:
    kmeans_k = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    clusters_k = kmeans_k.fit_predict(X_all)

    ari_k = adjusted_rand_score(y_all, clusters_k)
    nmi_k = normalized_mutual_info_score(y_all, clusters_k)

    cluster_results.append({
        "k": k,
        "ARI": ari_k,
        "NMI": nmi_k
    })

cluster_df = pd.DataFrame(cluster_results)
print("\nClustering quality for different k:")
print(cluster_df.to_string(index=False))

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_all_pca = pca.fit_transform(X_all)

print("Explained variance ratio:", pca.explained_varianceratio)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.scatterplot(
    x=X_all_pca[:, 0],
    y=X_all_pca[:, 1],
    hue=clusters,
    palette="tab10",
    s=10,
    alpha=0.7,
    legend=False
)
plt.title("KMeans clusters (k=2) në PCA-2D")
plt.xlabel("PC1")
plt.ylabel("PC2")

plt.subplot(1, 2, 2)
sns.scatterplot(
    x=X_all_pca[:, 0],
    y=X_all_pca[:, 1],
    hue=y_all,
    palette="Set1",
    s=10,
    alpha=0.7,
    legend=False
)
plt.title("Etiketat reale (0=normal, 1=attack) në PCA-2D")
plt.xlabel("PC1")
plt.ylabel("PC2")

plt.tight_layout()
plt.show()